# Ollama Feature Extraction

This notebook contains the `OllamaFeatureExtractor` class, which utilizes a local Ollama instance to extract product features from the product name.

In [1]:
import urllib.request
import json
import pandas as pd
import ssl

class OllamaFeatureExtractor:
    """
    A class to interact with a local Ollama instance to perform feature extraction.
    """
    def __init__(self, model_name: str = "llama3:latest", base_url: str = "http://127.0.0.1:11434"):
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{self.base_url}/api/generate"

    def extract_features(self, text: str) -> str:
        """
        Sends a prompt to the local Ollama model to extract features from a product name.
        """
        prompt = f"""You are a data extraction assistant. Extract the product features from the given product name and return them as clear key-value pairs formatted nicely.\nExample output:\ndisplay - 8 inch\nHD - yes\nNetwork - Wifi\ncolor - Magenta\n\nIMPORTANT: Output ONLY the key-value pairs separated by newlines. DO NOT output any introductory or concluding conversational text.\n\nProduct Name: {text}\nFeatures:"""
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }
        
        try:
            req = urllib.request.Request(self.api_url, json.dumps(data).encode('utf-8'))
            req.add_header('Content-Type', 'application/json')
            # Bypass potential proxy issues that cause hangs
            handler = urllib.request.ProxyHandler({})
            opener = urllib.request.build_opener(handler)
            with opener.open(req, timeout=120) as response:
                result = json.loads(response.read().decode())
                return result.get('response', '').strip()
        except Exception as e:
            return f"Error: {e}"
            
    def analyze_dataframe(self, df: pd.DataFrame, text_column: str, sample_size: int = 5) -> pd.DataFrame:
        """
        Analyzes a sample of the dataframe and prepares a result dataframe.
        """
        print(f"\n--- Extracting features from a sample of {sample_size} products with {self.model_name} ---", flush=True)
        sample_df = df.sample(n=sample_size, random_state=42) if len(df) > sample_size else df
        
        results = []
        for idx, row in sample_df.iterrows():
            text = str(row[text_column])
            print(f"\nProcessing Product: {text[:100]}...", flush=True)
            extracted_features = self.extract_features(text)
            
            results.append({
                'Product Name': text[:100] + '...' if len(text) > 100 else text,
                'Extracted Features': extracted_features
            })
            print(f"{extracted_features}", flush=True)
            print("-" * 40, flush=True)
            
        return pd.DataFrame(results)